# FROM 1D TO 2D

In summary:
* Use FFT to find dominant periods
* Convert each period into a 2D grid
* Columns = within-cycle behavior
* Rows = across-cycle behavior
* Now temporal patterns become spatial → easier to learn with CNNs

Step by step:

#### 0. Starting point: the 1D time series

They define:

$$
X_{1D} \in \mathbb{R}^{T \times C}
$$

* \(T\) = number of time steps
* \(C\) = number of variables (channels) (e.g. temperature, humiity...)

So:

* each row = time step
* each column = a variable

Example:

$$
X_{1D} =
\begin{bmatrix}
x_1^{(1)} & x_1^{(2)} \\
x_2^{(1)} & x_2^{(2)} \\
\vdots & \vdots \\
x_T^{(1)} & x_T^{(2)}
\end{bmatrix}
$$

#### 1. Find periodicities using FFT

They do:

$$
A = \text{Avg}(\text{Amp}(\text{FFT}(X_{1D})))
$$

Let’s unpack:

FFT: Transforms time → frequency: reveals **which periodic patterns exist**

Amp(FFT): Take magnitude: tells **how strong each frequency is**

Avg over channels: Since you have C variables: they average to get a **single spectrum**

So we obtain a 1D array:

$$
A \in \mathbb{R}^{T}
$$

where $A_j$ = strength of frequency $j$

#### 2. Select the dominant frequencies

We pick the **k strongest frequencies**:

$$
\{f_1, \dots, f_k\} = \text{argTopk}(A)
$$

In this way we ignore noise and keep only meaningful periodic patterns.

#### 3. Convert frequency → period

We obtain the correspondents $k$ period values:

$$
p_i = \left\lceil \frac{T}{f_i} \right\rceil, \quad i \in\ \{ 1, \dots, k \}
$$

* frequency = "how many cycles in T"
* period = "length of one cycle"

All the process seen till now to obtain $A$ and the frequencise/periods, we call it $\text{Period()}$ function.

#### 4. Key idea: reshape into 2D

Now comes the main step. For all frequencies $f_i$ and period $p_i$ (so for all $i \in\ \{ 1, \dots, k \}$), we reshape the same 1D original data to obtain $k$ different 2D matrixes of shape $(p_i,f_i)$ (or more correctly, in 3D tensors cause we have still the channel dimension $C$, so shape = $(p_i,f_i,C)$):

$$
X_{2D}^i = \text{Reshape}_{p_i, f_i}(\text{Padding}(X_{1D})), \quad i \in\ \{ 1, \dots, k \}
$$
$$
so \quad \{X_{2D}^1, \dots, X_{2D}^k\}
$$

So instead of 1D sequential data $[x_1, x_2, \dots, x_T]$ with dimension $X_{1D} \in \mathbb{R}^{T \times C}$, this operation reorganizes the temporal dimension $T$ into two axes:
- $p_i$: number of rows (row = intra-period variation (position inside cycle))
- $f_i$: number of columns (columns = inter-period variation (across cycles))

so we obtain a new matrix of shape $\text{(period length)} \times \text{(number of periods)}$ (so the indexing is $\text{(position inside period)} \times \text{(period index)}$) for each selected $k$-th period/frequency.

Even if indexed as '2D', each representation is not a matrix but a 3D tensor:

$$
X_{2D}^i \in \mathbb{R}^{p_i \times f_i \times C}
$$

Note: the channel dimension $C$ is still present cause we removed it by averaging over channels when we computed $A$ previously, but then we used $A$ only to identify the present frequencies, and nothing else; in fact, we proceeded with our steps using $X_{1D} \in \mathbb{R}^{T \times C}$ for our input in padding, reshaping etc. So, to be correct, we should say that ours $X^{2D}_i$ are 3D tensors (cause they are stacked 2D matrixes).

TLDR: we obtain $k$ different 2D representations of the same 1D time series, each corresponding to a different detected periodic structure


### Why padding is needed

Reshape only works if size matches. We need $T_{\text{padded}} = p_i \cdot f_i$, but in general $T \neq p_i \cdot f_i$ because $p_i = \lceil T / f_i \rceil$, so $p_i f_i \ge T$, usually strictly larger.

So what do they do? They extend:

$$
X_{1D} \rightarrow \text{Padding}(X_{1D})
$$

by adding zeros so that:

$$
T_{\text{padded}} = p_i \cdot f_i
$$

Simple example: Suppose:
* $T = 10$
* detected frequency → $f_i = 3$

Then:

$$
p_i = \lceil 10/3 \rceil = 4
$$

So:

$$
p_i f_i = 12
$$

We need to add 2 padded values (zeros) to our original sequential data 1D. This allows:

$$
\text{reshape } 10 \rightarrow 4 \times 3
$$

without losing structure.

---

### Example 

Suppose:

* \(T = 12\)
* detected period = \(p = 4\)

Then:

* number of cycles = \(f = T/p = 3\)

Original 1D:

$$
[x_1, x_2, x_3, x_4, x_5, x_6, x_7, x_8, x_9, x_{10}, x_{11}, x_{12}]
$$

Reshape into 2D:

$$
\begin{bmatrix}
x_1 & x_5 & x_9 \\
x_2 & x_6 & x_{10} \\
x_3 & x_7 & x_{11} \\
x_4 & x_8 & x_{12}
\end{bmatrix}
$$

Interpretation:

Columns (intra-period): Each column = one cycle:
* column 1: \(x_1, x_2, x_3, x_4\)
* column 2: \(x_5, x_6, x_7, x_8\)

→ variation **within a period**

Rows (inter-period): Each row = same position across cycles:
* row 1: \(x_1, x_5, x_9\)
* row 2: \(x_2, x_6, x_{10}\)

→ variation **across periods**

Why padding?

If \(T\) is not divisible by \(p_i\):

They do:

$$
\text{Padding}(X_{1D})
$$

→ add zeros so reshape works cleanly

Why multiple 2D tensors?

They don’t assume a single period.

They take:

$$
k \text{ dominant frequencies}
$$

So they get:

$$
\{X_{2D}^1, \dots, X_{2D}^k\}
$$

Each one corresponds to:

* a different periodic pattern

# TIMES BLOCK

## Global picture: what is a TimesBlock doing?

A **TimesBlock** is basically a single layer that does this loop:

1. take a 1D time series representation
2. discover its dominant periods (at *feature level*, not raw signal)
3. reshape it into multiple 2D “views” (one per period)
4. process each 2D view with a CNN-style block (Inception)
5. map everything back to 1D
6. merge results using learned importance weights
7. add residual connection

So the global formula is:
$$
X^{l}_{1D} = \text{TimesBlock} ( X^{l-1}_{1D} ) + X^{l-1}_{1D}
$$

So conceptually:

> “Turn time series into multiple images → process like images → collapse back → combine”

And this is repeated layer by layer.

## PHASE 1: CAPTURING TEMPORAL 2D VARIATIONS

#### 1.1 Input of the TimesBlock

For layer $l$, the input is:

$$
X^{l-1}_{1D} \in \mathbb{R}^{T \times d_{model}}
$$

where:

* $T$ = time length
* $d_{model}$ = feature dimension per time step

So each time step is “embedded”.

Even at first step we do:

$$
X^0_{1D} = \text{Embed}(X_{1D})
$$

#### 1.2 First key operation inside the block: Period detection

As seen before, using FFT they compute:

$$
A^{l-1}, \{f_1,\dots,f_k\}, \{p_1,\dots,p_k\} = \text{Period}(X^{l-1}_{1D})
$$

Interpretation:

* $f_i$ = dominant frequencies
* $p_i = \frac{T}{f_i}$ = corresponding periods
* $k$ = number of selected periodic patterns

Important: now periods are computed on **latent features**, not raw data. So the model refines its notion of periodicity layer by layer.

#### 1.3 Second step: build 2D representations

Again as seen before, for each period $i$ we obtain its 2D representation:

$$
X^{l,i}_{2D} = \text{Reshape}_{p_i, f_i}(\text{Padding}(X^{l-1}_{1D}))
$$

where $X^{l,i}_{2D}$ is the 2D representation obtained at layer $l$ for frequence $i$.

Input shape:

$$
X^{l-1}_{1D} \in \mathbb{R}^{T \times d_{model}}
$$

Output hape:

$$
X^{l,i}_{2D} \in \mathbb{R}^{p_i \times f_i \times d_{model}}
$$

So each slice is basically a 2D image where each pixel has a feature vector of dimension $d_{model}$.

#### 1.4 Inception block

They apply:

$$
\hat{X}^{l,i}_{2D} = \text{Inception}(X^{l,i}_{2D})
$$

This is the key “learning” part. Think of it as:
* a CNN with multiple kernel sizes (e.g. $1 \times 1$, $3 \times 3$, $5 \times 5$)
* applied over the 2D time-period grid

 What does that mean in time-series terms? Each kernel sees:
* horizontally → evolution across cycles (inter-period structure)
* vertically → structure inside a cycle (intra-period structure)

So instead of modeling $x_t \rightarrow x_{t+1}$ it models local patterns in a “time image”

Here input and output have same shape: $\mathbb{R}^{p_i \times f_i \times d_{model}}$


#### 1.5 Reshape back to 1D

They then flatten back:

$$
\hat{X}^{l,i}_{1D} = \text{Trunc}(\text{Reshape}_{1,(p_i \cdot f_i)}(\hat{X}^{l,i}_{2D}))
$$

Step-by-step:
1. flatten 2D grid back into sequence
2. reshape → length $p_i \cdot f_i$
3. truncate to original $T$

So:

$$
\hat{X}^{l,i}_{1D} \in \mathbb{R}^{T \times d_{model}}
$$

Same shape as the layer input.

Why truncation? Because padding may have extended the sequence, so they remove artificial values.

So now we have, for layer $l$:

$$
\hat{X}^{l,1}_{1D}, \hat{X}^{l,2}_{1D}, \dots, \hat{X}^{l,k}_{1D}, \quad \text{and each} \in \mathbb{R}^{T \times d_{model}}
$$

Each is the same input signal, but processed under a different notion of periodic structure

## PHASE 2: ADAPTIVE AGGREGATION

#### 2.1 Aggregation

We must now combine the $k$ obtained results after reshaping. We could do something like:

$$
X^l_{1D} = \sum_{i=1}^{k} w_i \hat{X}^{l,i}_{1D}
$$

But weights are not uniform, so which do we choose?

We use the fact that $A^{l-1}_{f_i}$ is the (scalar) strength of frequency $f_i$ from FFT, so then we use the Softmax function to turn them into probabilities:
$$
\hat{A}^{l-1}_{f_1}, \dots, \hat{A}^{l-1}_{f_k} = \text{Softmax}(A^{l-1}_{f_1}, \dots, A^{l-1}_{f_k})
$$

so we can define:

$$
w_i = \hat{A}^{l-1}_{f_i}
$$

and finally obtain:

$$
X^l_{1D} = \sum_{i=1}^{k} \hat{A}^{l-1}_{f_i} \hat{X}^{l,i}_{1D}
$$

Remember: the values $\hat{A}^{l-1}_{f_i}$ are scalars, while $\hat{X}^{l,i}_{1D} \in \mathbb{R}^{T \times d_{model}}$

So:
* strong frequency → more influence
* weak frequency → less influence



#### 2.2 Add skip connection

At the very top:

$$
X^l_{1D} = \text{TimesBlock}(X^{l-1}_{1D}) + X^{l-1}_{1D}
$$

So the full layer is “learn a transformation, then add the original signal back”

This is exactly ResNet logic.

## COMMENTS

After the reshape step, each time series (for each period) becomes something that looks like a **2D image with channels**. At that point, the authors make a conceptual leap:

> “If it looks like an image, we can process it with image models.”

The Inception block they used earlier is just one possible choice. But the authors emphasize that it is not fundamental.

You could replace it with:
* ResNet (classic CNN residual networks)
* ResNeXt (more expressive grouped convolutions)
* ConvNeXt (modern CNN design closer to Transformers)
* even attention-based vision models (ViTs or hybrid models)

So the time series problem is effectively converted into a vision problem on structured temporal images.

Even though many backbones are possible, they choose Inception for the main experiments because it is:
* parameter-efficient
* multi-scale (important for capturing patterns of different temporal sizes)
* computationally lighter than many modern backbones

So there is a trade-off:
* stronger models (ConvNeXt, attention) → potentially better accuracy
* Inception → better efficiency and still strong performance